# Stage 1 risk-score: Linear Regression

This notebook tunes the collision-level casualty-count regressor before the three classifier
notebooks are used. It creates out-of-fold `risk_score` values so the final ensemble does not
receive a prediction made by a regressor trained on the same collision. It is deliberately
not executed by the code review; run all cells on Windows after pushing.

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, KFold, cross_val_predict
import sys
sys.path.insert(0, str(Path.cwd() / 'hyper parameter')); sys.path.insert(0, str(Path.cwd()))
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
if not (ROOT / 'dataset').exists(): ROOT = ROOT.parent
TRAIN_PATH = ROOT / 'dataset' / 'processed' / 'regression_train.csv'
TEST_PATH = ROOT / 'dataset' / 'processed' / 'regression_test.csv'
RESULTS = ROOT / 'hyper parameter' / 'results' / 'linear_regression'
BEST = ROOT / 'hyper parameter' / 'best'
MODEL_DIR = ROOT / 'models' / 'saved_models'
RESULTS.mkdir(parents=True, exist_ok=True); BEST.mkdir(parents=True, exist_ok=True); MODEL_DIR.mkdir(parents=True, exist_ok=True)
SEED = 19
TARGET = 'number_of_casualties'
GROUP = 'collision_index'
FEATURES = ['number_of_vehicles', 'day_of_week', 'first_road_class', 'road_type', 'speed_limit', 'junction_detail', 'junction_control', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'urban_or_rural_area', 'special_conditions_at_site', 'carriageway_hazards']
NUMERIC = ['number_of_vehicles', 'speed_limit']
CATEGORICAL = [c for c in FEATURES if c not in NUMERIC]

In [2]:
train = pd.read_csv(TRAIN_PATH, dtype={GROUP: str})
test = pd.read_csv(TEST_PATH, dtype={GROUP: str})
X, y = train[FEATURES], train[TARGET].astype(float)
X_test, y_test = test[FEATURES], test[TARGET].astype(float)
preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL),
    ('numeric', StandardScaler(), NUMERIC),
], remainder='drop')
pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', LinearRegression())])
PARAM_GRID = {
    'regressor__fit_intercept': [True, False],
    'regressor__positive': [True, False],
}
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
print('collision-level train:', X.shape, 'test:', X_test.shape)
print('parameter combinations:', 4)

collision-level train: (81058, 13) test: (20467, 13)
parameter combinations: 4


In [3]:
scoring = {'r2': 'r2', 'rmse': 'neg_root_mean_squared_error', 'mae': 'neg_mean_absolute_error'}
search = RandomizedSearchCV(pipeline, PARAM_GRID, scoring=scoring, refit='r2', n_iter=4, random_state=SEED, cv=cv, n_jobs=1, verbose=2, return_train_score=True, error_score='raise')
search.fit(X, y)
cv_results = pd.DataFrame(search.cv_results_).sort_values('rank_test_r2')
cv_results.to_csv(RESULTS / 'cv_results.csv', index=False)
print('best parameters:', search.best_params_)
print('best CV R2:', search.best_score_)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
[CV] END regressor__fit_intercept=True, regressor__positive=True; total time=   1.3s
[CV] END regressor__fit_intercept=True, regressor__positive=True; total time=   1.2s
[CV] END regressor__fit_intercept=True, regressor__positive=True; total time=   1.2s
[CV] END regressor__fit_intercept=True, regressor__positive=True; total time=   1.2s
[CV] END regressor__fit_intercept=True, regressor__positive=True; total time=   1.1s
[CV] END regressor__fit_intercept=True, regressor__positive=False; total time=   0.4s
[CV] END regressor__fit_intercept=True, regressor__positive=False; total time=   0.4s
[CV] END regressor__fit_intercept=True, regressor__positive=False; total time=   0.4s
[CV] END regressor__fit_intercept=True, regressor__positive=False; total time=   0.5s
[CV] END regressor__fit_intercept=True, regressor__positive=False; total time=   0.5s
[CV] END regressor__fit_intercept=False, regressor__positive=True; total time=   1.2s

In [4]:
best_model = search.best_estimator_
oof = cross_val_predict(best_model, X, y, cv=cv, method='predict', n_jobs=1)
test_prediction = best_model.predict(X_test)
train_oof = pd.DataFrame({GROUP: train[GROUP], 'risk_score': oof})
test_scores = pd.DataFrame({GROUP: test[GROUP], 'risk_score': test_prediction})
train_oof.to_csv(RESULTS / 'oof_predictions.csv', index=False)
test_scores.to_csv(RESULTS / 'test_predictions.csv', index=False)
metrics = {
    'model': 'linear_regression', 'features': FEATURES, 'best_params': search.best_params_,
    'cv_r2': float(search.best_score_), 'oof_rmse': float(np.sqrt(mean_squared_error(y, oof))),
    'oof_mae': float(mean_absolute_error(y, oof)), 'test_r2': float(r2_score(y_test, test_prediction)),
    'test_rmse': float(np.sqrt(mean_squared_error(y_test, test_prediction))),
    'test_mae': float(mean_absolute_error(y_test, test_prediction)),
}
(RESULTS / 'best_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
joblib.dump(best_model, BEST / 'linear_regression_best.pkl')
joblib.dump(best_model, MODEL_DIR / 'linear_regression_1.pkl')
(BEST / 'linear_regression_best_params.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print(json.dumps(metrics, indent=2))

{
  "model": "linear_regression",
  "features": [
    "number_of_vehicles",
    "day_of_week",
    "first_road_class",
    "road_type",
    "speed_limit",
    "junction_detail",
    "junction_control",
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "urban_or_rural_area",
    "special_conditions_at_site",
    "carriageway_hazards"
  ],
  "best_params": {
    "regressor__positive": true,
    "regressor__fit_intercept": false
  },
  "cv_r2": 0.07290336910191877,
  "oof_rmse": 0.6727028494126462,
  "oof_mae": 0.3920850899416641,
  "test_r2": 0.018442115354421973,
  "test_rmse": 1.16208275071257,
  "test_mae": 0.3878520016918195
}
